# 面试问题：LLM 连续批处理下怎样让随机采样可复放，并隔离请求间 RNG？

**一句话回答。** 将采样随机数定义为 request id、生成 step、seed 和 sampling config 的纯函数，而不是消费全局 RNG；记录模型/tokenizer/参数版本。这样调度顺序不会改变受控 sampler 的结果，但浮点 kernel、硬件与模型服务仍可能破坏位级确定性。

本 Notebook 仅用 Python 标准库手写数据合同、核心算法和失败分支；受控样例用于验证不变量，不代表生产吞吐、模型质量或硬件精度。

**资料入口。** [vLLM Reproducibility 文档](https://docs.vllm.ai/en/v0.9.1/usage/reproducibility.html) 明确说明默认不保证复现；本例只验证请求级 RNG 合同。


In [ ]:
question = "batch-invariant sampling"  # 执行本行的状态、计算或校验逻辑。
assert "sampling" in question  # 执行本行的状态、计算或校验逻辑。
assert 1 + 1 == 2  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 采样配置本身是生成输入的一部分

同一个 prompt 仅记录 seed 不够：temperature、top-p、top-k、logit bias、模型 revision 和 tokenizer revision 都会改变候选分布。这里用二元概率模拟已完成的 logit 处理。


In [ ]:
requests = {"a": [[0.7, 0.3], [0.4, 0.6]], "b": [[0.2, 0.8], [0.6, 0.4]]}  # 执行本行的状态、计算或校验逻辑。
config = {"seed": 19, "temperature": 0.8, "model": "m1", "tokenizer": "t1"}  # 执行本行的状态、计算或校验逻辑。
assert set(requests) == {"a", "b"}  # 执行本行的状态、计算或校验逻辑。
assert config["temperature"] > 0  # 执行本行的状态、计算或校验逻辑。
assert config["model"] == "m1"  # 执行本行的状态、计算或校验逻辑。

## 2. 请求级坐标 RNG：不消费共享状态

教学用确定性整数混合函数把 request id 与 step 映射到 [0,1)。它不是密码学 RNG；生产使用成熟 RNG，但接口仍应避免由批次插入顺序决定随机数消费。


In [ ]:
def coordinate_uniform(seed, request_id, step):  # 执行本行的状态、计算或校验逻辑。
    identity = sum((index + 1) * ord(char) for index, char in enumerate(request_id))  # 执行本行的状态、计算或校验逻辑。
    mixed = (seed * 1103515245 + identity * 12345 + step * 2654435761) & 0xffffffff  # 执行本行的状态、计算或校验逻辑。
    return mixed / 4294967296.0  # 执行本行的状态、计算或校验逻辑。
assert coordinate_uniform(19, "a", 0) == coordinate_uniform(19, "a", 0)  # 执行本行的状态、计算或校验逻辑。
assert coordinate_uniform(19, "a", 0) != coordinate_uniform(19, "a", 1)  # 执行本行的状态、计算或校验逻辑。
assert 0 <= coordinate_uniform(19, "b", 0) < 1  # 执行本行的状态、计算或校验逻辑。

## 3. 从 CDF 采样并处理边界

给定已归一化概率和 uniform，累积概率首次覆盖 uniform 的 token 即为样本。真实服务还要在此前完成 temperature、top-p、bad words、grammar mask 等 logit 处理，并记录最终 sampling parameters。


In [ ]:
def sample(probabilities, uniform):  # 执行本行的状态、计算或校验逻辑。
    cumulative = 0.0  # 执行本行的状态、计算或校验逻辑。
    for token, probability in enumerate(probabilities):  # 执行本行的状态、计算或校验逻辑。
        cumulative += probability  # 执行本行的状态、计算或校验逻辑。
        if uniform < cumulative:  # 执行本行的状态、计算或校验逻辑。
            return token  # 执行本行的状态、计算或校验逻辑。
    return len(probabilities) - 1  # 执行本行的状态、计算或校验逻辑。
assert sample([0.7, 0.3], 0.0) == 0  # 执行本行的状态、计算或校验逻辑。
assert sample([0.7, 0.3], 0.95) == 1  # 执行本行的状态、计算或校验逻辑。
assert sample([0.7, 0.3], 1.0) == 1  # 执行本行的状态、计算或校验逻辑。

## 4. 调度重排不应影响同一请求的 token 流

每一轮调度可按到达、deadline 或 KV 预算重排；在该合同下，每个请求每个 step 只读取自己的 coordinate RNG。因此先跑 a 再跑 b 与反序跑应完全相同。


In [ ]:
def coordinate_schedule(order):  # 执行本行的状态、计算或校验逻辑。
    output = {request_id: [] for request_id in order}  # 执行本行的状态、计算或校验逻辑。
    for step in range(2):  # 执行本行的状态、计算或校验逻辑。
        for request_id in order:  # 执行本行的状态、计算或校验逻辑。
            uniform = coordinate_uniform(config["seed"], request_id, step)  # 执行本行的状态、计算或校验逻辑。
            output[request_id].append(sample(requests[request_id][step], uniform))  # 执行本行的状态、计算或校验逻辑。
    return output  # 执行本行的状态、计算或校验逻辑。
forward_order = coordinate_schedule(["a", "b"])  # 执行本行的状态、计算或校验逻辑。
reverse_order = coordinate_schedule(["b", "a"])  # 执行本行的状态、计算或校验逻辑。
assert forward_order["a"] == reverse_order["a"]  # 执行本行的状态、计算或校验逻辑。
assert forward_order["b"] == reverse_order["b"]  # 执行本行的状态、计算或校验逻辑。
assert len(forward_order["a"]) == 2  # 执行本行的状态、计算或校验逻辑。

## 5. 反例：共享全局 RNG 会把排队顺序变成模型行为

若每个 active request 依次消费同一条随机数流，插入/取消一个请求会改变其他请求拿到的 uniform。即使 logits 相同，这也会让 A/B 测试和 replay 难以解释。


In [ ]:
def global_schedule(order):  # 执行本行的状态、计算或校验逻辑。
    uniforms = iter([0.1, 0.9, 0.2, 0.8])  # 执行本行的状态、计算或校验逻辑。
    output = {request_id: [] for request_id in order}  # 执行本行的状态、计算或校验逻辑。
    for step in range(2):  # 执行本行的状态、计算或校验逻辑。
        for request_id in order:  # 执行本行的状态、计算或校验逻辑。
            output[request_id].append(sample([0.5, 0.5], next(uniforms)))  # 执行本行的状态、计算或校验逻辑。
    return output  # 执行本行的状态、计算或校验逻辑。
global_forward = global_schedule(["a", "b"])  # 执行本行的状态、计算或校验逻辑。
global_reverse = global_schedule(["b", "a"])  # 执行本行的状态、计算或校验逻辑。
assert global_forward["a"] != global_reverse["a"]  # 执行本行的状态、计算或校验逻辑。
assert global_forward["a"] == [0, 0]  # 执行本行的状态、计算或校验逻辑。
assert global_reverse["a"] == [1, 1]  # 执行本行的状态、计算或校验逻辑。

## 6. trace 要保存足以解释、而非重放秘密的字段

request id、step、seed、模型/Tokenizer revision、采样参数 fingerprint 和 token id 能定位调度/配置回归。提示词和工具结果是否保存需要单独的隐私策略；trace 不等于无条件记录原文。


In [ ]:
def trace_row(request_id, step, token):  # 执行本行的状态、计算或校验逻辑。
    return {"id": request_id, "step": step, "token": token, "seed": config["seed"], "model": config["model"], "tokenizer": config["tokenizer"]}  # 执行本行的状态、计算或校验逻辑。
trace = trace_row("a", 1, forward_order["a"][1])  # 执行本行的状态、计算或校验逻辑。
assert trace["step"] == 1  # 执行本行的状态、计算或校验逻辑。
assert trace["seed"] == 19  # 执行本行的状态、计算或校验逻辑。
assert set(trace) == {"id", "step", "token", "seed", "model", "tokenizer"}  # 执行本行的状态、计算或校验逻辑。

## 7. 版本变化必须显式拒绝伪 replay

同一 seed 在模型权重、tokenizer、logit processor 或 kernel 变化后不应被称为同一次可复放执行。这里把比较做成确定性 gate；生产还需固定硬件/框架版本并用统计稳定性而非迷信位级一致。


In [ ]:
def replayable(trace_value, current):  # 执行本行的状态、计算或校验逻辑。
    return trace_value["model"] == current["model"] and trace_value["tokenizer"] == current["tokenizer"] and trace_value["seed"] == current["seed"]  # 执行本行的状态、计算或校验逻辑。
assert replayable(trace, config)  # 执行本行的状态、计算或校验逻辑。
assert not replayable(trace, {**config, "model": "m2"})  # 执行本行的状态、计算或校验逻辑。
assert not replayable(trace, {**config, "seed": 20})  # 执行本行的状态、计算或校验逻辑。

## 8. 验收同时覆盖隔离性与随机性质量

最小回归集至少测 batch-order invariance、取消/插入请求、同 seed 可复放、不同 seed 可变化及版本 gate。它只覆盖 sampler；模型前向的非确定性、浮点规约和分布式 kernel 还需要单独环境控制。


In [ ]:
assert coordinate_schedule(["a", "b"]) == coordinate_schedule(["a", "b"])  # 执行本行的状态、计算或校验逻辑。
assert coordinate_uniform(20, "a", 0) != coordinate_uniform(19, "a", 0)  # 执行本行的状态、计算或校验逻辑。
assert all(token in (0, 1) for tokens in forward_order.values() for token in tokens)  # 执行本行的状态、计算或校验逻辑。
assert global_forward != global_reverse  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试回答应说清“batch-invariant sampler”只隔离 RNG 消费，不自动保证整条 LLM 推理位级确定。它需要与模型/tokenizer/参数版本、确定性 kernel 配置、trace 和统计回归共同组成可审计的复现策略。
